In [ ]:
from pathlib import Path
import os
from os import PathLike
from typing import Optional, Sequence, Union
from fastcore.basics import patch

from trouver.obsidian.vault import NoteDoesNotExistError, VaultNote, note_name_from_path, all_paths_to_notes_in_vault, NoteNotFoundInCacheError, NoteNotUniqueError, NotePathIsNotIdentifiedError
from trouver.obsidian.links import ObsidianLink, LinkType, replace_links_in_text
from trouver.obsidian.vault import path_to_obs_id

In [ ]:
import shutil
import tempfile
from unittest import mock

from fastcore.test import *
from nbdev.showdoc import show_doc

from trouver.helper.tests import _test_directory

In [ ]:
#| export obsidian.vault
# TODO: test/document
@patch
def path(self: VaultNote,
        relative=False # If `True`, then return the path relative to the vault. Otherwise, return the absolute path.
        ) -> Union[Path, None]: # Path to the note if self.rel_path is determined. `None` otherwise.
    r"""Returns the path to the note.

    **Raises**
    - NotePathIsNotIdentifiedError
        - If the relative path of `self` is not identified.
    """
    if not self.rel_path_identified():
        raise NotePathIsNotIdentifiedError.from_note(self)
    return Path(self.rel_path) if relative\
        else self.vault / self.rel_path

In [ ]:
show_doc(VaultNote.path)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L816){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.path

>      VaultNote.path (relative=False)

*Returns the path to the note.

**Raises**
- NotePathIsNotIdentifiedError
    - If the relative path of `self` is not identified.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| relative | bool | False | If `True`, then return the path relative to the vault. Otherwise, return the absolute path. |
| **Returns** | **Optional** |  | **Path to the note if self.rel_path is determined. `None` otherwise.** |

In [ ]:
#| export obsidian.vault
@patch
# TODO: test/document
def directory(self: VaultNote,
              relative=False # If `True`, then return the path of the directory relative to the vault.
              ) -> Path: # The path of the directory that the note is in.
    r"""Return the directory that the note is in.
    """
    rel_dir = Path(os.path.dirname(self.rel_path))
    return rel_dir if relative else self.vault / rel_dir

In [ ]:
show_doc(VaultNote.directory)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L833){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.directory

>      VaultNote.directory (relative=False)

*Return the directory that the note is in.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| relative | bool | False | If `True`, then return the path of the directory relative to the vault. |
| **Returns** | **Path** |  | **The path of the directory that the note is in.** |

#### By name

The most convenient way to construct an existing `VaultNote` is to specify the vault in which it exists and the note's name, assuming that the note of the specified name exists and is unique in the specified vault. We can also verify the existence of the file corresponding to a `VaultNote` objet with the `.exists` method.

In [ ]:
#| export obsidian.vault
@patch
def exists(
        self: VaultNote,
        update_cache=False # If `True`, then update the cache and try to identify `self.rel_path` before verifying whether the note exists in the vault.
        ) -> bool:
    r"""Returns `True` if `self.rel_path` is identified and
    if the note exists in the vault.
    
    Setting `update_cache` to `True` updates the cache before verifying
    whether the `VaultNote` object exists if the `VaultNote` object is
    specified by `name` and not `rel_path`. Doing so guarantees that the
    output is correct at the possible cost of runtime.
    """
    if self.rel_path is None:
        if update_cache:
            self.identify_rel_path(update_cache=True)
        else:
            return False
    try:
        abs_path = self.path()        
    except NotePathIsNotIdentifiedError as e:
        return False
    return os.path.exists(abs_path)

In [ ]:
show_doc(VaultNote.exists)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L869){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.exists

>      VaultNote.exists (update_cache=False)

*Returns `True` if `self.rel_path` is identified and
if the note exists in the vault.

Setting `update_cache` to `True` updates the cache before verifying
whether the `VaultNote` object exists if the `VaultNote` object is
specified by `name` and not `rel_path`. Doing so guarantees that the
output is correct at the possible cost of runtime.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| update_cache | bool | False | If `True`, then update the cache and try to identify `self.rel_path` before verifying whether the note exists in the vault. |
| **Returns** | **bool** |  |  |

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    vault_note = VaultNote(temp_vault, name='exponential_function')
    assert vault_note.exists()
    print(vault_note.name)
    print(f'`vault_note` is located, relative to `vault`, at {vault_note.rel_path}.')

exponential_function
`vault_note` is located, relative to `vault`, at analysis\exponential_function.md.


#### By relative path

Alternatively, a `VaultNote` object can be created by passing an argument to the `rel_path` parameter. In this case, the note of the specified path does not need to exist.

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    vault_note = VaultNote(temp_vault, rel_path='non_existent_folder/non_existent_note.md')
    assert not vault_note.exists()
    # Note that there is not a unique note of name `ring`.
    vault_note = VaultNote(temp_vault, rel_path='algebra/ring.md')  
    assert vault_note.exists()

The `rel_path` parameter takes precedence over the `name` parameter.

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    vault_note = VaultNote(temp_vault, rel_path='non_existent_folder/non_existent_note.md', name='ring')
    assert vault_note.name == 'non_existent_note'

If the arguments for both the `name` and the `rel_path` parameters are `None`, then a `ValueError` is raised:



In [ ]:
test_vault = _test_directory() / 'test_vault_1'
with ExceptionExpected(ValueError):
    vault_note = VaultNote(test_vault, rel_path=None, name=None)

#### Identifying the `VaultNote` object

In [ ]:
show_doc(VaultNote.identify_rel_path)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.identify_rel_path

>      VaultNote.identify_rel_path (update_cache=False)

*Sets `self.rel_path` to a path, if not already done so.

If `self.rel_path` is not already set as a path, then the cache
is searched to find a note whose name is `self.name` (which is
necessarily specified).*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| update_cache | bool | False | If `True`, if the cache is searched, and if a note of the specified name is not found in the cache, then the cache is updated and searched again. Defaults to `False`. |
| **Returns** | **None** |  |  |

In [ ]:
show_doc(VaultNote.rel_path_identified)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.rel_path_identified

>      VaultNote.rel_path_identified ()

*Return `True` if `self.rel_path` is identified, i.e. is a path
that is not `None`.*

When the `name` parameter (as opposed to the `rel_path` parameter) is specified in the constructor of a `VaultNote` object, the constructor looks into the cache of the `VaultNote` class to identify a note with the specified name in the specified vault. If the cache contains no such note, then the cache is updated and searched again. If the cache still contains no such note, then the `rel_path` attribute of the `VaultNote` object is left unidentified (i.e. is set to `None`). 

Assuming that a note of the specified name is created later, the relative path of the `VaultNote` object can be identified using the `identify_rel_path(update_cache=True)` method. Moreover, the `rel_path_identified` method returns `True` if the relative path is identified.

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    vn = VaultNote(temp_vault, name='does_not_exist_at_first')
    assert not vn.rel_path_identified()
    # Create a note at the root of the vault.
    open(temp_vault / 'does_not_exist_at_first.md', 'w').close()
    vn.identify_rel_path(update_cache=True)
    assert vn.rel_path_identified()
    assert vn.exists()
    test_eq(vn.rel_path, 'does_not_exist_at_first.md')
    

## Getting information about the note

In [ ]:
#| export obsidian.vault
@patch
def obsidian_identifier(self: VaultNote) -> str:
    r"""Return the Obsidian identifier of the `VaultNote` object.
    
    This is the note's unqiue Obsidian id in the vault. This is like a
    path str with forward slashes `/` (as opposed to backwards `\` slashes)
    and without a file extension (`.md`).
    """
    return path_to_obs_id(self.rel_path)

In [ ]:
show_doc(VaultNote.obsidian_identifier)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L894){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.obsidian_identifier

>      VaultNote.obsidian_identifier ()

*Return the Obsidian identifier of the `VaultNote` object.

This is the note's unqiue Obsidian id in the vault. This is like a
path str with forward slashes `/` (as opposed to backwards `\` slashes)
and without a file extension (`.md`).*

Here are some convenient ways to get information about the `VaultNote`:

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    vault_note = VaultNote(temp_vault, name='exponential_function')
    print(f'Obsidian vault identifier:\t{vault_note.obsidian_identifier()}')
    print(f'relative path:\t{vault_note.rel_path}')
    print(f'Part of the absolute path of the note:\t{str(vault_note.path(relative=False))[:7]}')
    print(f'note name:\t{vault_note.name}')
    print(f'directory that the note is in relative to the vault:\t{vault_note.directory(relative=True)}')

Obsidian vault identifier:	analysis/exponential_function
relative path:	analysis\exponential_function.md
Part of the absolute path of the note:	c:\User
note name:	exponential_function
directory that the note is in relative to the vault:	analysis


The `VaultNote.path` method raises a `NotePathIsNotIdentifiedError` if the `VaultNote` object's relative path is not idetnfied:

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    vn = VaultNote(temp_vault, name='does_not_exist')
    with ExceptionExpected(NotePathIsNotIdentifiedError):
        vn.path()